In [1]:
# Import core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configure visualization
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

print("🌍 TESS TOI Exoplanet Habitability Classification Pipeline")
print("📊 Libraries loaded successfully!")
print("="*60)

🌍 TESS TOI Exoplanet Habitability Classification Pipeline
📊 Libraries loaded successfully!


# 1. Data Loading and Initial Exploration

Load the TESS TOI dataset and examine its structure for habitability-relevant features.

In [2]:
# Load TESS TOI dataset
print("📂 Loading TESS TOI dataset...\n")
df = pd.read_csv(
    "../Datasets/TOI_dataset.csv",
    sep=",",
    comment="#",
    skipinitialspace=True,
    low_memory=False
)

# Strip whitespace from column names
df.columns = df.columns.str.strip()

# Dataset summary
print("✅ TESS TOI Dataset successfully loaded!\n")
print(f"📊 Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"📝 Total features: {df.shape[1]}\n")

# Show column categories
print("🔍 Available Column Categories:")
print(f"  • Identifiers: toi, tid, toipfx")
print(f"  • Planetary params: pl_orbper, pl_rade, pl_insol, pl_eqt")
print(f"  • Stellar params: st_teff, st_rad, st_mass, st_tmag")
print(f"  • Transit params: pl_trandurh, pl_trandep, pl_tranmid")

print("\n📋 First 3 rows preview:")
df.head(3)

📂 Loading TESS TOI dataset...

✅ TESS TOI Dataset successfully loaded!

📊 Shape: 7668 rows × 87 columns
📝 Total features: 87

🔍 Available Column Categories:
  • Identifiers: toi, tid, toipfx
  • Planetary params: pl_orbper, pl_rade, pl_insol, pl_eqt
  • Stellar params: st_teff, st_rad, st_mass, st_tmag
  • Transit params: pl_trandurh, pl_trandep, pl_tranmid

📋 First 3 rows preview:


,rowid,toi,toipfx,tid,ctoi_alias,pl_pnum,tfopwg_disp,rastr,ra,raerr1,...,st_loggerr2,st_logglim,st_loggsymerr,st_rad,st_raderr1,st_raderr2,st_radlim,st_radsymerr,toi_created,rowupdate
0,1,1000.01,1000,50365310,5.036531e+07,1,FP,07h29m25.85s,112.357708,NaN,...,-0.07,0,1,2.16986,0.072573,-0.072573,0,1,2019-07-24 15:58:33,2024-09-09 10:08:01
1,2,1001.01,1001,88863718,8.886372e+07,1,PC,08h10m19.31s,122.580465,NaN,...,-0.09,0,1,2.01000,0.090000,-0.090000,0,1,2019-07-24 15:58:33,2023-04-03 14:31:04
2,3,1002.01,1002,124709665,1.247097e+08,1,FP,06h58m54.47s,104.726966,NaN,...,NaN,0,1,5.73000,NaN,NaN,0,1,2019-07-24 15:58:33,2022-07-11 16:02:02


In [3]:
# Check original disposition distribution
print("🎯 Original TFOPWG Disposition Distribution:")
if 'tfopwg_disp' in df.columns:
    print(df['tfopwg_disp'].value_counts())
    print(f"\nTotal confirmed planets (CP): {(df['tfopwg_disp'] == 'CP').sum()}")
    print(f"Total candidates (PC): {(df['tfopwg_disp'] == 'PC').sum()}")
    print(f"Total false positives (FP): {(df['tfopwg_disp'] == 'FP').sum()}")
else:
    print("⚠️ 'tfopwg_disp' column not found")

print("\n" + "="*60)
print("📊 Data Quality Assessment:")
print("="*60)

# Missing values analysis
missing_counts = df.isnull().sum()
missing_pct = (missing_counts / len(df)) * 100

print(f"Columns with missing values: {(missing_counts > 0).sum()}")
print(f"Columns with >50% missing: {(missing_pct > 50).sum()}")
print(f"Columns with >90% missing: {(missing_pct > 90).sum()}")

# Data types
print(f"\nNumeric columns: {df.select_dtypes(include=[np.number]).shape[1]}")
print(f"Categorical columns: {df.select_dtypes(include=['object']).shape[1]}")

🎯 Original TFOPWG Disposition Distribution:
tfopwg_disp
PC     4675
FP     1192
CP      679
KP      565
APC     459
FA       98
Name: count, dtype: int64

Total confirmed planets (CP): 679
Total candidates (PC): 4675
Total false positives (FP): 1192

📊 Data Quality Assessment:
Columns with missing values: 48
Columns with >50% missing: 12
Columns with >90% missing: 12

Numeric columns: 82
Categorical columns: 5


# 2. Habitability Classification Criteria

Define the criteria for classifying TESS TOI exoplanets into habitability categories.

In [4]:
print("="*60)
print("🌍 HABITABILITY CLASSIFICATION CRITERIA")
print("="*60)

print("\n📚 Classification Categories:\n")

print("1️⃣ POTENTIALLY_HABITABLE (Earth-like conditions)")
print("   • Planet Radius: 0.5 - 2.0 Earth radii (Rocky planets)")
print("   • Insolation Flux: 0.25 - 4.0 Earth flux (Conservative HZ)")
print("   • Equilibrium Temp: 180K - 310K (Liquid water range)")
print("   • Orbital Period: 10 - 500 days (Reasonable year length)")

print("\n2️⃣ HABITABILITY_ZONE (In HZ but not Earth-like)")
print("   • In habitable zone by insolation (0.25 - 4.0)")
print("   • But fails other criteria (too large, wrong temp, etc.)")
print("   • Requires further investigation")

print("\n3️⃣ NON_HABITABLE (Outside habitable parameters)")
print("   • Too hot (>400K) or too cold (<150K)")
print("   • Gas giants (>4 Earth radii)")
print("   • Extreme insolation (<0.1 or >10 Earth flux)")
print("   • Very short (<5 days) or very long (>1000 days) periods")

print("\n🔬 Key TESS TOI Parameters for Habitability:")
print("  ✓ pl_rade: Planet radius in Earth radii")
print("  ✓ pl_insol: Insolation flux in Earth flux")
print("  ✓ pl_eqt: Equilibrium temperature in Kelvin")
print("  ✓ pl_orbper: Orbital period in days")
print("  ✓ st_teff: Stellar temperature (star type matters!)")

print("\n" + "="*60)

🌍 HABITABILITY CLASSIFICATION CRITERIA

📚 Classification Categories:

1️⃣ POTENTIALLY_HABITABLE (Earth-like conditions)
   • Planet Radius: 0.5 - 2.0 Earth radii (Rocky planets)
   • Insolation Flux: 0.25 - 4.0 Earth flux (Conservative HZ)
   • Equilibrium Temp: 180K - 310K (Liquid water range)
   • Orbital Period: 10 - 500 days (Reasonable year length)

2️⃣ HABITABILITY_ZONE (In HZ but not Earth-like)
   • In habitable zone by insolation (0.25 - 4.0)
   • But fails other criteria (too large, wrong temp, etc.)
   • Requires further investigation

3️⃣ NON_HABITABLE (Outside habitable parameters)
   • Too hot (>400K) or too cold (<150K)
   • Gas giants (>4 Earth radii)
   • Extreme insolation (<0.1 or >10 Earth flux)
   • Very short (<5 days) or very long (>1000 days) periods

🔬 Key TESS TOI Parameters for Habitability:
  ✓ pl_rade: Planet radius in Earth radii
  ✓ pl_insol: Insolation flux in Earth flux
  ✓ pl_eqt: Equilibrium temperature in Kelvin
  ✓ pl_orbper: Orbital period in days


# 3. Data Cleaning & Preprocessing

Clean the dataset and prepare for habitability classification.

In [5]:
print("="*60)
print("🧹 DATA CLEANING PHASE")
print("="*60)

# Create working copy
df_clean = df.copy()
initial_shape = df_clean.shape

print(f"\n🔄 Starting with: {initial_shape[0]} rows × {initial_shape[1]} columns\n")

# 1. Filter to confirmed planets and strong candidates
if 'tfopwg_disp' in df_clean.columns:
    print("1️⃣ Filtering to confirmed planets and strong candidates...")
    initial_count = len(df_clean)
    # Keep CP (Confirmed Planet) and PC (Planet Candidate)
    df_clean = df_clean[df_clean['tfopwg_disp'].isin(['CP', 'PC'])]
    filtered_count = len(df_clean)
    print(f"   Kept {filtered_count} confirmed + candidates (removed {initial_count - filtered_count} false positives)")
else:
    print("1️⃣ No disposition column - keeping all planets")

# 2. Identify critical habitability features
print("\n2️⃣ Identifying critical habitability features...")

critical_habitability_features = {
    'pl_rade': 'Planet Radius [Earth Radius]',
    'pl_insol': 'Insolation Flux [Earth Flux]',
    'pl_eqt': 'Equilibrium Temperature [K]',
    'pl_orbper': 'Orbital Period [days]',
    'st_teff': 'Stellar Temperature [K]',
    'st_rad': 'Stellar Radius [Solar Radius]',
    'st_mass': 'Stellar Mass [Solar Mass]',
    'st_tmag': 'TESS Magnitude'
}

# Check which features are available
available_critical = [f for f in critical_habitability_features.keys() if f in df_clean.columns]
missing_critical = [f for f in critical_habitability_features.keys() if f not in df_clean.columns]

print(f"   Available critical features: {len(available_critical)}/{len(critical_habitability_features)}")
for feat in available_critical:
    missing_pct = (df_clean[feat].isnull().sum() / len(df_clean)) * 100
    print(f"   ✓ {feat}: {missing_pct:.1f}% missing")

if missing_critical:
    print(f"\n   ⚠️ Missing features: {missing_critical}")

# 3. Handle missing values strategically
print(f"\n3️⃣ Handling missing values...")

# Drop rows with missing critical habitability parameters
before_drop = len(df_clean)

# Require at least pl_rade and pl_orbper
essential_features = ['pl_rade', 'pl_orbper']
available_essential = [f for f in essential_features if f in df_clean.columns]

if available_essential:
    df_clean = df_clean.dropna(subset=available_essential)
    after_drop = len(df_clean)
    print(f"   Dropped {before_drop - after_drop} rows missing essential features")
else:
    print("   ⚠️ Essential features not available - keeping all rows")
    after_drop = before_drop

# 4. Fill remaining missing values with median
print(f"\n4️⃣ Imputing remaining missing values...")

numeric_features = df_clean.select_dtypes(include=[np.number]).columns
for col in numeric_features:
    if df_clean[col].isnull().sum() > 0:
        median_val = df_clean[col].median()
        filled_count = df_clean[col].isnull().sum()
        df_clean[col].fillna(median_val, inplace=True)
        if filled_count > 0 and filled_count < 100:  # Only show if not too many
            print(f"   Filled {filled_count} missing values in {col} with median")

# 5. Remove extreme outliers
print(f"\n5️⃣ Handling extreme outliers...")

if 'pl_rade' in df_clean.columns:
    before_outlier = len(df_clean)
    df_clean = df_clean[(df_clean['pl_rade'] > 0.1) & (df_clean['pl_rade'] < 50)]
    removed_outliers = before_outlier - len(df_clean)
    if removed_outliers > 0:
        print(f"   Removed {removed_outliers} planets with extreme radius values")

final_shape = df_clean.shape
print(f"\n✅ Cleaning completed!")
print(f"📊 Final dataset: {final_shape[0]} rows × {final_shape[1]} columns")
print(f"📉 Total removed: {initial_shape[0] - final_shape[0]} rows ({((initial_shape[0] - final_shape[0])/initial_shape[0]*100):.1f}%)")

print("\n" + "="*60)

🧹 DATA CLEANING PHASE

🔄 Starting with: 7668 rows × 87 columns

1️⃣ Filtering to confirmed planets and strong candidates...
   Kept 5354 confirmed + candidates (removed 2314 false positives)

2️⃣ Identifying critical habitability features...
   Available critical features: 7/8
   ✓ pl_rade: 6.4% missing
   ✓ pl_insol: 2.1% missing
   ✓ pl_eqt: 3.4% missing
   ✓ pl_orbper: 1.4% missing
   ✓ st_teff: 2.1% missing
   ✓ st_rad: 6.4% missing
   ✓ st_tmag: 0.0% missing

   ⚠️ Missing features: ['st_mass']

3️⃣ Handling missing values...
   Dropped 418 rows missing essential features

4️⃣ Imputing remaining missing values...
   Filled 28 missing values in st_pmra with median
   Filled 28 missing values in st_pmraerr1 with median
   Filled 28 missing values in st_pmraerr2 with median
   Filled 28 missing values in st_pmralim with median
   Filled 28 missing values in st_pmrasymerr with median
   Filled 28 missing values in st_pmdec with median
   Filled 28 missing values in st_pmdecerr1 with m

# 4. Habitability Classification

Apply habitability criteria to classify each TESS TOI exoplanet.

In [6]:
print("="*60)
print("🌍 APPLYING HABITABILITY CLASSIFICATION")
print("="*60)

# Initialize habitability column
df_clean['habitability_class'] = 'UNKNOWN'

# Get required columns with fallback values
pl_rade = df_clean['pl_rade'] if 'pl_rade' in df_clean.columns else pd.Series([999]*len(df_clean))
pl_insol = df_clean['pl_insol'] if 'pl_insol' in df_clean.columns else pd.Series([999]*len(df_clean))
pl_eqt = df_clean['pl_eqt'] if 'pl_eqt' in df_clean.columns else pd.Series([999]*len(df_clean))
pl_orbper = df_clean['pl_orbper'] if 'pl_orbper' in df_clean.columns else pd.Series([999]*len(df_clean))

print("\n1️⃣ Classifying as POTENTIALLY_HABITABLE...")
print("   Criteria: Earth-like size, HZ insolation, temperate conditions")

potentially_habitable = (
    (pl_rade >= 0.5) & (pl_rade <= 2.0) &  # Earth-like radius
    (pl_insol >= 0.25) & (pl_insol <= 4.0) &  # Conservative habitable zone
    (pl_eqt >= 180) & (pl_eqt <= 310) &  # Liquid water temperature range
    (pl_orbper >= 10) & (pl_orbper <= 500)  # Reasonable orbital period
)

df_clean.loc[potentially_habitable, 'habitability_class'] = 'POTENTIALLY_HABITABLE'
ph_count = potentially_habitable.sum()
print(f"   ✓ Identified {ph_count} potentially habitable planets ({ph_count/len(df_clean)*100:.2f}%)")

print("\n2️⃣ Classifying as HABITABILITY_ZONE...")
print("   Criteria: In HZ by insolation but not meeting all Earth-like criteria")

habitability_zone = (
    (df_clean['habitability_class'] == 'UNKNOWN') &
    (
        ((pl_insol >= 0.25) & (pl_insol <= 4.0)) |
        ((pl_eqt >= 200) & (pl_eqt <= 350))
    )
)

df_clean.loc[habitability_zone, 'habitability_class'] = 'HABITABILITY_ZONE'
hz_count = habitability_zone.sum()
print(f"   ✓ Identified {hz_count} planets in habitability zone ({hz_count/len(df_clean)*100:.2f}%)")

print("\n3️⃣ Classifying remaining as NON_HABITABLE...")
print("   Criteria: Outside habitable parameters")

df_clean.loc[df_clean['habitability_class'] == 'UNKNOWN', 'habitability_class'] = 'NON_HABITABLE'
nh_count = (df_clean['habitability_class'] == 'NON_HABITABLE').sum()
print(f"   ✓ Identified {nh_count} non-habitable planets ({nh_count/len(df_clean)*100:.2f}%)")

print("\n" + "="*60)
print("📊 HABITABILITY CLASSIFICATION RESULTS")
print("="*60)

class_distribution = df_clean['habitability_class'].value_counts()
print("\n🎯 Final Distribution:")
for cls, count in class_distribution.items():
    pct = (count / len(df_clean)) * 100
    print(f"   {cls}: {count} planets ({pct:.2f}%)")

print(f"\n✅ All {len(df_clean)} planets classified!")
print("="*60)

🌍 APPLYING HABITABILITY CLASSIFICATION

1️⃣ Classifying as POTENTIALLY_HABITABLE...
   Criteria: Earth-like size, HZ insolation, temperate conditions
   ✓ Identified 10 potentially habitable planets (0.20%)

2️⃣ Classifying as HABITABILITY_ZONE...
   Criteria: In HZ by insolation but not meeting all Earth-like criteria
   ✓ Identified 149 planets in habitability zone (3.02%)

3️⃣ Classifying remaining as NON_HABITABLE...
   Criteria: Outside habitable parameters
   ✓ Identified 4776 non-habitable planets (96.78%)

📊 HABITABILITY CLASSIFICATION RESULTS

🎯 Final Distribution:
   NON_HABITABLE: 4776 planets (96.78%)
   HABITABILITY_ZONE: 149 planets (3.02%)
   POTENTIALLY_HABITABLE: 10 planets (0.20%)

✅ All 4935 planets classified!


# 5. Feature Engineering & ML Preparation

Create habitability-focused features and prepare data for machine learning.

In [7]:
print("="*60)
print("🔧 FEATURE ENGINEERING FOR HABITABILITY")
print("="*60)

df_features = df_clean.copy()

print("\n1️⃣ Creating derived habitability features...\n")

# 1. Earth Similarity Index components
if 'pl_rade' in df_features.columns:
    df_features['radius_similarity'] = 1 - abs(df_features['pl_rade'] - 1.0) / 10
    df_features['radius_similarity'] = df_features['radius_similarity'].clip(0, 1)
    print("   ✓ Created radius_similarity")

if 'pl_insol' in df_features.columns:
    df_features['insol_similarity'] = 1 - abs(df_features['pl_insol'] - 1.0) / 10
    df_features['insol_similarity'] = df_features['insol_similarity'].clip(0, 1)
    print("   ✓ Created insol_similarity")

if 'pl_eqt' in df_features.columns:
    df_features['temp_similarity'] = 1 - abs(df_features['pl_eqt'] - 255) / 500
    df_features['temp_similarity'] = df_features['temp_similarity'].clip(0, 1)
    print("   ✓ Created temp_similarity")

# 2. Habitability flags
if 'pl_insol' in df_features.columns:
    df_features['in_hz_conservative'] = ((df_features['pl_insol'] >= 0.25) & 
                                         (df_features['pl_insol'] <= 4.0)).astype(int)
    print("   ✓ Created habitable zone flag")

if 'pl_rade' in df_features.columns:
    df_features['is_rocky'] = (df_features['pl_rade'] <= 2.0).astype(int)
    df_features['is_earth_sized'] = ((df_features['pl_rade'] >= 0.8) & 
                                     (df_features['pl_rade'] <= 1.25)).astype(int)
    print("   ✓ Created planet type flags")

# 3. Logarithmic transformations
log_features = ['pl_orbper', 'pl_insol']
for feat in log_features:
    if feat in df_features.columns:
        df_features[f'{feat}_log'] = np.log1p(df_features[feat])
        print(f"   ✓ Created {feat}_log")

# 4. TESS-specific features
if 'pl_trandep' in df_features.columns and 'pl_trandurh' in df_features.columns:
    df_features['depth_duration_ratio'] = df_features['pl_trandep'] / (df_features['pl_trandurh'] + 1e-6)
    print("   ✓ Created depth_duration_ratio")

if 'st_tmag' in df_features.columns:
    df_features['tmag_bright'] = (df_features['st_tmag'] < 10).astype(int)
    print("   ✓ Created tmag_bright flag (bright host star)")

print(f"\n✅ Feature engineering completed!")
print(f"📊 Features after engineering: {df_features.shape[1]} columns")
print("="*60)

🔧 FEATURE ENGINEERING FOR HABITABILITY

1️⃣ Creating derived habitability features...

   ✓ Created radius_similarity
   ✓ Created insol_similarity
   ✓ Created temp_similarity
   ✓ Created habitable zone flag
   ✓ Created planet type flags
   ✓ Created pl_orbper_log
   ✓ Created pl_insol_log
   ✓ Created depth_duration_ratio
   ✓ Created tmag_bright flag (bright host star)

✅ Feature engineering completed!
📊 Features after engineering: 98 columns


In [8]:
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
import pickle
import os

print("="*60)
print("🤖 PREPARING ML-READY DATASET")
print("="*60)

# Select numeric features for ML
numeric_cols = df_features.select_dtypes(include=[np.number]).columns.tolist()

# Exclude ID and metadata columns
exclude_cols = ['rowid', 'toi', 'tid', 'tfopwg_disp', 'ra', 'dec']
ml_features = [f for f in numeric_cols if not any(ex in str(f).lower() for ex in exclude_cols)]

print(f"\n1️⃣ Selected {len(ml_features)} numeric features for ML")

# Handle infinite values and NaNs
df_features[ml_features] = df_features[ml_features].replace([np.inf, -np.inf], np.nan)
for col in ml_features:
    if df_features[col].isnull().sum() > 0:
        df_features[col].fillna(df_features[col].median(), inplace=True)

# Prepare X and y
X = df_features[ml_features].copy()
y = df_features['habitability_class'].copy()

print(f"\n2️⃣ Dataset shapes:")
print(f"   Feature matrix (X): {X.shape}")
print(f"   Target vector (y): {y.shape}")

# Encode target variable
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print(f"\n3️⃣ Class encoding:")
for i, class_name in enumerate(le.classes_):
    count = (y_encoded == i).sum()
    print(f"   {class_name} → {i} ({count} samples)")

# Feature scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns, index=X.index)

minmax_scaler = MinMaxScaler()
X_minmax = minmax_scaler.fit_transform(X)
X_minmax_df = pd.DataFrame(X_minmax, columns=X.columns, index=X.index)

print(f"\n4️⃣ Applied StandardScaler and MinMaxScaler")

# Train/val/test split
X_train, X_temp, y_train, y_temp = train_test_split(
    X_scaled_df, y_encoded, test_size=0.4, random_state=42, stratify=y_encoded
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"\n5️⃣ Data splits:")
print(f"   Training: {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"   Validation: {X_val.shape[0]} samples ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"   Testing: {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.1f}%)")

print("\n✅ ML-ready dataset prepared!")
print("="*60)

🤖 PREPARING ML-READY DATASET

1️⃣ Selected 44 numeric features for ML

2️⃣ Dataset shapes:
   Feature matrix (X): (4935, 44)
   Target vector (y): (4935,)

3️⃣ Class encoding:
   HABITABILITY_ZONE → 0 (149 samples)
   NON_HABITABLE → 1 (4776 samples)
   POTENTIALLY_HABITABLE → 2 (10 samples)

4️⃣ Applied StandardScaler and MinMaxScaler

5️⃣ Data splits:
   Training: 2961 samples (60.0%)
   Validation: 987 samples (20.0%)
   Testing: 987 samples (20.0%)

✅ ML-ready dataset prepared!


# 6. Save Processed Data

Save the cleaned, classified, and ML-ready TESS TOI habitability dataset.

In [9]:
print("="*60)
print("💾 SAVING TESS TOI HABITABILITY DATASETS")
print("="*60)

# Create TESS-specific directories
os.makedirs('../data/processed/tess', exist_ok=True)
os.makedirs('../artifacts/tess', exist_ok=True)

print("\n1️⃣ Saving train/validation/test splits...\n")

# Save StandardScaler splits
datasets = {
    'train': (X_train, y_train),
    'val': (X_val, y_val),
    'test': (X_test, y_test)
}

for split_name, (X_split, y_split) in datasets.items():
    split_df = X_split.copy()
    split_df['target'] = y_split
    split_df['target_name'] = le.inverse_transform(y_split)
    
    filepath = f'../data/processed/tess/tess_habitability_{split_name}.csv'
    split_df.to_csv(filepath, index=False)
    print(f"   ✓ Saved {filepath} ({split_df.shape[0]} rows)")

# Save MinMax splits
X_train_mm, X_temp_mm, y_train_mm, y_temp_mm = train_test_split(
    X_minmax_df, y_encoded, test_size=0.4, random_state=42, stratify=y_encoded
)
X_val_mm, X_test_mm, y_val_mm, y_test_mm = train_test_split(
    X_temp_mm, y_temp_mm, test_size=0.5, random_state=42, stratify=y_temp_mm
)

datasets_mm = {
    'train_minmax': (X_train_mm, y_train_mm),
    'val_minmax': (X_val_mm, y_val_mm),
    'test_minmax': (X_test_mm, y_test_mm)
}

for split_name, (X_split, y_split) in datasets_mm.items():
    split_df = X_split.copy()
    split_df['target'] = y_split
    split_df['target_name'] = le.inverse_transform(y_split)
    
    filepath = f'../data/processed/tess/tess_habitability_{split_name}.csv'
    split_df.to_csv(filepath, index=False)
    print(f"   ✓ Saved {filepath}")

# Save full processed dataset
print(f"\n2️⃣ Saving full processed dataset...")
full_processed_path = '../data/processed/tess/tess_habitability_full_processed.csv'
df_features.to_csv(full_processed_path, index=False)
print(f"   ✓ Saved {full_processed_path}")

# Save transformers
print(f"\n3️⃣ Saving ML artifacts...")

transformers = {
    'standard_scaler': scaler,
    'minmax_scaler': minmax_scaler,
    'label_encoder': le
}

for name, transformer in transformers.items():
    filepath = f'../artifacts/tess/tess_habitability_{name}.pkl'
    with open(filepath, 'wb') as f:
        pickle.dump(transformer, f)
    print(f"   ✓ Saved {filepath}")

# Save metadata
metadata = {
    'dataset_name': 'TESS TOI Exoplanet Habitability',
    'classification_type': 'habitability',
    'feature_names': ml_features,
    'n_features': len(ml_features),
    'target_classes': le.classes_.tolist(),
    'n_classes': len(le.classes_),
    'total_samples': len(df_features),
    'split_sizes': {
        'train': len(X_train),
        'val': len(X_val),
        'test': len(X_test)
    },
    'class_distribution': df_features['habitability_class'].value_counts().to_dict()
}

metadata_path = '../artifacts/tess/tess_habitability_metadata.pkl'
with open(metadata_path, 'wb') as f:
    pickle.dump(metadata, f)
print(f"   ✓ Saved {metadata_path}")

print("\n" + "="*60)
print("✅ ALL TESS TOI HABITABILITY DATASETS SAVED!")
print("="*60)

💾 SAVING TESS TOI HABITABILITY DATASETS

1️⃣ Saving train/validation/test splits...

   ✓ Saved ../data/processed/tess/tess_habitability_train.csv (2961 rows)
   ✓ Saved ../data/processed/tess/tess_habitability_val.csv (987 rows)
   ✓ Saved ../data/processed/tess/tess_habitability_test.csv (987 rows)
   ✓ Saved ../data/processed/tess/tess_habitability_train_minmax.csv
   ✓ Saved ../data/processed/tess/tess_habitability_val_minmax.csv
   ✓ Saved ../data/processed/tess/tess_habitability_test_minmax.csv

2️⃣ Saving full processed dataset...
   ✓ Saved ../data/processed/tess/tess_habitability_full_processed.csv

3️⃣ Saving ML artifacts...
   ✓ Saved ../artifacts/tess/tess_habitability_standard_scaler.pkl
   ✓ Saved ../artifacts/tess/tess_habitability_minmax_scaler.pkl
   ✓ Saved ../artifacts/tess/tess_habitability_label_encoder.pkl
   ✓ Saved ../artifacts/tess/tess_habitability_metadata.pkl

✅ ALL TESS TOI HABITABILITY DATASETS SAVED!


# 7. Final Summary

Summary of TESS TOI habitability processing.

In [10]:
print("="*60)
print("🌍 TESS TOI EXOPLANET HABITABILITY - FINAL SUMMARY")
print("="*60)

print("\n📊 DATASET STATISTICS")
print("-"*60)
print(f"Original dataset: {initial_shape[0]} rows × {initial_shape[1]} columns")
print(f"Processed dataset: {df_features.shape[0]} rows × {df_features.shape[1]} columns")
print(f"Data reduction: {((initial_shape[0] - df_features.shape[0])/initial_shape[0]*100):.1f}% rows removed")

print("\n🎯 HABITABILITY CLASSIFICATION RESULTS")
print("-"*60)
final_dist = df_features['habitability_class'].value_counts()
for cls, count in final_dist.items():
    pct = (count / len(df_features)) * 100
    print(f"{cls}: {count} planets ({pct:.2f}%)")

print("\n🔧 FEATURE ENGINEERING")
print("-"*60)
print(f"Total ML features: {len(ml_features)}")
print("Key features: Earth similarity indices, HZ flags, planet types")

print("\n💾 OUTPUT FILES")
print("-"*60)
print("✓ Train/Val/Test splits (StandardScaler & MinMaxScaler)")
print("✓ Full processed dataset")
print("✓ Scalers and encoders")
print("✓ Metadata")

print("\n🔍 PROJECT STATUS")
print("-"*60)
print("✅ K2 dataset processed (01_k2_habitability.ipynb)")
print("✅ Kepler dataset processed (02_kepler_habitability.ipynb)")
print("✅ TESS TOI dataset processed (03_tess_toi_habitability.ipynb)")
print("\n📈 READY FOR MACHINE LEARNING PHASE!")
print("Next: Develop ML models and ensemble predictions")

print("\n" + "="*60)
print("✅ TESS TOI HABITABILITY PROCESSING COMPLETE!")
print("🚀 All three missions processed and ready!")
print("="*60)

🌍 TESS TOI EXOPLANET HABITABILITY - FINAL SUMMARY

📊 DATASET STATISTICS
------------------------------------------------------------
Original dataset: 7668 rows × 87 columns
Processed dataset: 4935 rows × 98 columns
Data reduction: 35.6% rows removed

🎯 HABITABILITY CLASSIFICATION RESULTS
------------------------------------------------------------
NON_HABITABLE: 4776 planets (96.78%)
HABITABILITY_ZONE: 149 planets (3.02%)
POTENTIALLY_HABITABLE: 10 planets (0.20%)

🔧 FEATURE ENGINEERING
------------------------------------------------------------
Total ML features: 44
Key features: Earth similarity indices, HZ flags, planet types

💾 OUTPUT FILES
------------------------------------------------------------
✓ Train/Val/Test splits (StandardScaler & MinMaxScaler)
✓ Full processed dataset
✓ Scalers and encoders
✓ Metadata

🔍 PROJECT STATUS
------------------------------------------------------------
✅ K2 dataset processed (01_k2_habitability.ipynb)
✅ Kepler dataset processed (02_kepler_hab